# Import Environment variables

In [1]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=temporary-workspace-bucket, bucketName=temporary-workspace-bucket-wb-perky-cabbage-8342
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: gs://temporary-workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
dataset_test2 BQ_DATASET
prep_C2024Q3R9 BQ_DATASET
rw-mig

In [2]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

WORKSPACE_CDR = wb-silky-artichoke-2408.C2024Q3R9
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
GOOGLE_PROJECT = wb-perky-cabbage-8342
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.R
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.sas


# Librairy

In [3]:
import os
import numpy as np
import pandas as pd
from google.cloud import bigquery

# Data's import

## Clinical data

In [4]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "df_54_RiskFactors_CurrentSmoking.tsv"

df = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

In [5]:
df.columns

Index(['person_id', 'inclusion_date', 'first_breast_cancer_date', 'delay_days',
       'age_at_inclusion', 'has_bc', 'eur_rye', 'eas_rye', 'amr_rye',
       'afr_rye', 'sas_rye', 'mid_rye', 'dominant_origin', 'ancestry_80',
       'biopsy_result', 'age_category', 'race_us_bcsc', 'race_us_bcsc_detail',
       'breast_cancer_family_history_first_degree', 'tobacco_survey_date',
       'tobacco_answer_concept_id', 'tobacco_answer_label', 'smoking_status'],
      dtype='object')

# Contextual informations

__Sept facteurs de risque de cancer du sein ont été pris en compte :__

* l’âge des premières règles ;
* la parité (antécédents d’accouchement) ;
* l’âge à la première grossesse à terme ;
* l’indice de masse corporelle (IMC) à l’âge adulte chez les femmes ménopausées ;
* la taille à l’âge adulte ;
* le recours actuel à un traitement hormonal de la ménopause (THM) à base d’œstrogènes et de progestérone ;
* la consommation moyenne d’alcool au cours de la vie.

__Harmonisation des données et définitions des variables__

* _Les variables dépendantes du temps ont été évaluées à une date de référence définie comme la date du diagnostic pour les cas et la date de l'entretien pour les témoins dans les études cas-témoins. [..] la date de référence était celle du dernier questionnaire de suivi, si disponible ; sinon, la date du questionnaire initial a été utilisée._

* _En l'absence de données sur le statut ménopausique, nous avons utilisé l'âge médian (54 ans) comme indicateur de substitution : les femmes âgées de moins de 54 ans ont été considérées comme préménopausiques et celles âgées de 54 ans ou plus comme postménopausiques._

* _Le recours actuel à un THM à base d’œstrogènes et de progestérone a été défini comme un recours dans les six mois précédant la date de référence._

* _Dans les études cas-témoins, l’IMC a été calculé à partir du poids habituel à l’âge adulte ou du poids un an avant la date de référence, si cette donnée était disponible. Si cette variable n’était pas disponible, le poids au début de l’âge adulte a été utilisé comme indicateur. Le poids déclaré au moment du diagnostic ou lors de l’entretien dans les études cas-témoins n’a pas été utilisé afin d’éviter l’influence de la maladie sur le poids. Pour les deux études de cohorte prospectives (MCCS, UKBGS), nous avons utilisé le poids déclaré lors de l'entretien initial (avant le diagnostic)._

* _Les variables continues (âge des premières règles, AFTP, consommation d'alcool, taille et IMC) ont été modélisées à la fois comme des variables continues et catégorielles_

Source : [Associations conjointes d'un score de risque polygénique et de facteurs de risque environnementaux pour le cancer du sein dans le Breast Cancer Association Consortium](https://pmc.ncbi.nlm.nih.gov/articles/PMC5913605/#sec16)

# Lifetime intake of alcohol

## Data import

In [6]:
import os
import numpy as np
import pandas as pd
from google.cloud import bigquery

dataset = os.environ["WORKSPACE_CDR"]
client = bigquery.Client()

query_alcool_clean = f"""
SELECT 
    obs.person_id,
    obs.observation_date AS date_releve,
    obs.observation_source_concept_id AS id_question,
    obs.observation_source_value AS code_question,
    obs.value_source_concept_id AS id_reponse,
    c_src_r.concept_name AS libelle_reponse
FROM `{dataset}.observation` obs
LEFT JOIN `{dataset}.concept` c_src_r 
  ON obs.value_source_concept_id = c_src_r.concept_id

WHERE obs.observation_source_concept_id IN (
    1586198, -- Alcool au cours de la vie (Oui / Non)
    1586201, -- Fréquence de consommation au cours de l'année écoulée
    1586207  -- Nombre moyen de verres les jours de consommation
)
ORDER BY obs.person_id, obs.observation_date ASC
"""

df_alcool_brut = client.query(query_alcool_clean).to_dataframe()

print(f"Nombre total de lignes extraites : {len(df_alcool_brut)}")
print(
    f"Nombre de participantes uniques : {df_alcool_brut['person_id'].nunique()}"
)

df_alcool_brut

Nombre total de lignes extraites : 1517197
Nombre de participantes uniques : 578451


,person_id,date_releve,id_question,code_question,id_reponse,libelle_reponse
0,1000000,2019-08-29,1586201,Alcohol_DrinkFrequencyPastYear,1586202,Drink Frequency Past Year: Never
1,1000000,2019-08-29,1586198,Alcohol_AlcoholParticipant,1586199,Alcohol Participant: Yes
2,1000004,2019-07-26,1586201,Alcohol_DrinkFrequencyPastYear,1586205,Drink Frequency Past Year: 2 to 3 Per Week
3,1000004,2019-07-26,1586198,Alcohol_AlcoholParticipant,1586199,Alcohol Participant: Yes
4,1000004,2019-07-26,1586207,Alcohol_AverageDailyDrinkCount,1586208,Average Daily Drink Count: 1 or 2
...,...,...,...,...,...,...
1517192,9999860,2023-04-12,1586198,Alcohol_AlcoholParticipant,1586199,Alcohol Participant: Yes
1517193,9999860,2023-04-12,1586207,Alcohol_AverageDailyDrinkCount,1586208,Average Daily Drink Count: 1 or 2
1517194,9999996,2023-04-27,1586201,Alcohol_DrinkFrequencyPastYear,1586203,Drink Frequency Past Year: Monthly Or Less
1517195,9999996,2023-04-27,1586198,Alcohol_AlcoholParticipant,1586199,Alcohol Participant: Yes


## Calculation of the rate in g/day

__Résumé du traitement de conversion en g/jour__

* __Numérisation des modalités (Étape 1 & 2) :__ Il convertit les catégories qualitatives des questions `1586201` (fréquence) et `1586207` (quantité) en valeurs numériques moyennes (jours par semaine et verres par jour).
* __Alignement par date (Étape 3 & 4) :__ Il réunit les deux réponses pour chaque participante à une même date de relevé et attribue automatiquement $0\text{ verre}$ aux non-consommatrices (Jamais).
* __Calcul de l'exposition continue (Étape 5) :__ Il applique la formule d'éthanol pur ($14\text{ g/verre US}$) lissée sur $7\text{ jours}$ pour dériver la variable continue en g/jour :$$\text{alcool\_g\_jour} = \frac{\text{jours\_semaine} \times \text{verres\_jour\_conso} \times 14}{7}$$
* __Nettoyage :__ Il supprime les relevés partiels (sans calcul possible) et trie la table finale par identifiant et chronologie.

In [7]:
# 1. Cartographie numérique des modalités
map_frequence = {
    1586202: 0.0,  # Jamais -> 0 jour/semaine
    1586203: 0.25,  # Mensuelle ou moins -> ~1 jour/mois (0.25 j/sem)
    1586204: 0.75,  # 2 à 4 fois par mois -> ~3 jours/mois (0.75 j/sem)
    1586205: 2.5,  # 2 à 3 fois par semaine -> 2.5 jours/semaine
    1586206: 5.5,  # 4 fois ou plus par semaine -> 5.5 jours/semaine
}

map_quantite = {
    1586208: 1.5,  # 1 ou 2 verres -> 1.5
    1586209: 3.5,  # 3 ou 4 verres -> 3.5
    1586210: 5.5,  # 5 ou 6 verres -> 5.5
    1586211: 8.0,  # 7 à 9 verres -> 8.0
    1586212: 11.0,  # 10 verres ou plus -> 11.0
}

# 2. Isolement des sous-questions
df_freq = df_alcool_brut[df_alcool_brut["id_question"] == 1586201].copy()
df_freq["jours_semaine"] = df_freq["id_reponse"].map(map_frequence)

df_qty = df_alcool_brut[df_alcool_brut["id_question"] == 1586207].copy()
df_qty["verres_jour_conso"] = df_qty["id_reponse"].map(map_quantite)

# 3. Pivot et fusion par participante et date de relevé
df_alcool_pivoted = pd.merge(
    df_freq[["person_id", "date_releve", "jours_semaine"]],
    df_qty[["person_id", "date_releve", "verres_jour_conso"]],
    on=["person_id", "date_releve"],
    how="outer",
)

# 4. Traitement des non-consommatrices (Jamais -> 0 verre les jours de conso)
df_alcool_pivoted["verres_jour_conso"] = np.where(
    df_alcool_pivoted["jours_semaine"] == 0.0,
    0.0,
    df_alcool_pivoted["verres_jour_conso"],
)

# 5. Calcul de l'exposition moyenne en g/jour : (Jours/sem * Verres/jour * 14g) / 7 jours
df_alcool_pivoted["lifetime_intake_of_alcohol"] = (
    df_alcool_pivoted["jours_semaine"]
    * df_alcool_pivoted["verres_jour_conso"]
    * 14.0
) / 7.0
df_alcool_pivoted["lifetime_intake_of_alcohol"] = df_alcool_pivoted["lifetime_intake_of_alcohol"].round(
    1
)

# Nettoyage des lignes incomplètes et tri
df_alcool_final = df_alcool_pivoted.dropna(
    subset=["lifetime_intake_of_alcohol"]
).sort_values(["person_id", "date_releve"])

print(
    f"Nombre total de femmes avec calcul d'alcool valide : {df_alcool_final['person_id'].nunique()}"
)

df_alcool_final

Nombre total de femmes avec calcul d'alcool valide : 498314


,person_id,date_releve,jours_semaine,verres_jour_conso,lifetime_intake_of_alcohol
0,1000000,2019-08-29,0.00,0.0,0.0
1,1000004,2019-07-26,2.50,1.5,7.5
2,1000005,2018-06-19,0.75,3.5,5.2
3,1000012,2018-08-22,0.75,1.5,2.2
4,1000033,2019-07-26,0.25,1.5,0.8
...,...,...,...,...,...
518485,9999678,2023-04-17,2.50,3.5,17.5
518486,9999715,2023-05-31,0.00,0.0,0.0
518487,9999755,2023-03-08,2.50,1.5,7.5
518488,9999860,2023-04-12,0.25,1.5,0.8


### Verification of the calculation for an individual

In [8]:
df_alcool_final[df_alcool_final['person_id'] == 1000004]

,person_id,date_releve,jours_semaine,verres_jour_conso,lifetime_intake_of_alcohol
1,1000004,2019-07-26,2.5,1.5,7.5


In [9]:
df_alcool_brut[df_alcool_brut['person_id'] == 1000004]

,person_id,date_releve,id_question,code_question,id_reponse,libelle_reponse
2,1000004,2019-07-26,1586201,Alcohol_DrinkFrequencyPastYear,1586205,Drink Frequency Past Year: 2 to 3 Per Week
3,1000004,2019-07-26,1586198,Alcohol_AlcoholParticipant,1586199,Alcohol Participant: Yes
4,1000004,2019-07-26,1586207,Alcohol_AverageDailyDrinkCount,1586208,Average Daily Drink Count: 1 or 2


## Add column `alcohol_intake_category`

In [10]:
# 1. Définition des bornes (bins) et des libellés (labels)
bins = [-np.inf, 0.00001, 5.0, 15.0, 25.0, 35.0, 45.0, np.inf]

labels = [
    "< 0.00001 g/day",
    ">= 0.00001 to < 5 g/day",
    ">= 5 to < 15 g/day",
    ">= 15 to < 25 g/day",
    ">= 25 to < 35 g/day",
    ">= 35 to < 45 g/day",
    ">= 45 g/day",
]

# 2. Création de la colonne catégorielle dans df_alcool_final
df_alcool_final["alcohol_intake_category"] = pd.cut(
    df_alcool_final["lifetime_intake_of_alcohol"],
    bins=bins,
    labels=labels,
    right=False,  # Exclut la borne supérieure pour respecter le '<'
)

# Affichage de la répartition pour contrôle
print(df_alcool_final["alcohol_intake_category"].value_counts(sort=False))

df_alcool_final

alcohol_intake_category
< 0.00001 g/day             92869
>= 0.00001 to < 5 g/day    242848
>= 5 to < 15 g/day          78807
>= 15 to < 25 g/day         54565
>= 25 to < 35 g/day          4220
>= 35 to < 45 g/day         15646
>= 45 g/day                  9359
Name: count, dtype: int64


,person_id,date_releve,jours_semaine,verres_jour_conso,lifetime_intake_of_alcohol,alcohol_intake_category
0,1000000,2019-08-29,0.00,0.0,0.0,< 0.00001 g/day
1,1000004,2019-07-26,2.50,1.5,7.5,>= 5 to < 15 g/day
2,1000005,2018-06-19,0.75,3.5,5.2,>= 5 to < 15 g/day
3,1000012,2018-08-22,0.75,1.5,2.2,>= 0.00001 to < 5 g/day
4,1000033,2019-07-26,0.25,1.5,0.8,>= 0.00001 to < 5 g/day
...,...,...,...,...,...,...
518485,9999678,2023-04-17,2.50,3.5,17.5,>= 15 to < 25 g/day
518486,9999715,2023-05-31,0.00,0.0,0.0,< 0.00001 g/day
518487,9999755,2023-03-08,2.50,1.5,7.5,>= 5 to < 15 g/day
518488,9999860,2023-04-12,0.25,1.5,0.8,>= 0.00001 to < 5 g/day


## Merge with breast cancer cohort

In [11]:
df_alcohol_consumption = df.merge(df_alcool_final, on='person_id', how='left')
df_alcohol_consumption

,person_id,inclusion_date,first_breast_cancer_date,delay_days,age_at_inclusion,has_bc,eur_rye,eas_rye,amr_rye,afr_rye,...,breast_cancer_family_history_first_degree,tobacco_survey_date,tobacco_answer_concept_id,tobacco_answer_label,smoking_status,date_releve,jours_semaine,verres_jour_conso,lifetime_intake_of_alcohol,alcohol_intake_category
0,1700611,2019-09-17,NaN,NaN,70.255989,0,0.091868,0.000000,0.000000,0.848348,...,0,2019-09-17,1585858.0,100 Cigs Lifetime: Yes,1.0,2019-09-17,0.00,0.0,0.0,< 0.00001 g/day
1,1356439,2019-03-04,NaN,NaN,69.716632,0,0.065172,0.000000,0.025094,0.000000,...,0,2019-03-04,1585859.0,100 Cigs Lifetime: No,0.0,NaT,NaN,NaN,NaN,NaN
2,3451057,2023-01-18,NaN,NaN,71.594798,0,0.103268,0.000000,0.037591,0.000000,...,0,2023-01-18,1585859.0,100 Cigs Lifetime: No,0.0,NaT,NaN,NaN,NaN,NaN
3,1559161,2019-03-11,NaN,NaN,67.737166,0,0.006018,0.006429,0.965105,0.000000,...,0,2019-03-14,1585859.0,100 Cigs Lifetime: No,0.0,2019-03-14,0.25,1.5,0.8,>= 0.00001 to < 5 g/day
4,1468526,2019-10-07,NaN,NaN,63.310062,0,0.896282,0.000000,0.032661,0.000000,...,0,2019-10-07,1585858.0,100 Cigs Lifetime: Yes,1.0,NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124896,4386725,2023-05-11,NaN,NaN,40.903491,0,0.894660,0.000000,0.045839,0.000000,...,0,2023-05-11,1585859.0,100 Cigs Lifetime: No,0.0,2023-05-11,0.75,1.5,2.2,>= 0.00001 to < 5 g/day
124897,2743914,2022-08-08,NaN,NaN,40.147844,0,0.980679,0.000000,0.000000,0.000000,...,0,2022-08-08,1585859.0,100 Cigs Lifetime: No,0.0,2022-08-08,2.50,1.5,7.5,>= 5 to < 15 g/day
124898,1475516,2022-07-06,NaN,NaN,40.057495,0,0.848439,0.000000,0.049838,0.000000,...,0,2022-07-06,1585859.0,100 Cigs Lifetime: No,0.0,2022-07-06,0.75,1.5,2.2,>= 0.00001 to < 5 g/day
124899,9915233,2022-09-14,NaN,NaN,40.249144,0,0.673153,0.000000,0.050596,0.000000,...,0,2022-09-14,1585859.0,100 Cigs Lifetime: No,0.0,2022-09-14,0.00,0.0,0.0,< 0.00001 g/day


In [12]:
df_alcohol_consumption.value_counts(['has_bc','alcohol_intake_category'])

has_bc  alcohol_intake_category
0       >= 0.00001 to < 5 g/day    54659
        < 0.00001 g/day            21211
        >= 5 to < 15 g/day         13722
        >= 15 to < 25 g/day        10306
        >= 35 to < 45 g/day         2516
        >= 45 g/day                 1059
        >= 25 to < 35 g/day          406
1       >= 0.00001 to < 5 g/day      320
        < 0.00001 g/day              135
        >= 5 to < 15 g/day            94
        >= 15 to < 25 g/day           89
        >= 35 to < 45 g/day           14
        >= 45 g/day                    6
        >= 25 to < 35 g/day            1
Name: count, dtype: int64

# Data's export

In [14]:
destination_filename = 'Datas/df_54_RiskFactors_CurrentSmoking_AlcoholConsumption.tsv'
df_alcohol_consumption.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./Datas/df_54_RiskFactors_CurrentSmoking_AlcoholConsumption.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/ 28.0 MiB]                                                \r/ [1 files][ 28.0 MiB/ 28.0 MiB]                                                \r\nOperation completed over 1 objects/28.0 MiB.                                     \n'